# RAG Pipeline with Weaviate, BGE and Llama 3
---
In this notebook, we move from a **naive Retrieval-Augmented Generation (RAG)** pipeline to a more scalable architecture based on a **vector database with [Weaviate API](https://weaviate.io/)**. The pipeline relies on **BBC News data** as the knowledge base for retrieval. Relevant documents are fetched through **Semantic Search**, **BM25**, and **Hybrid Search**, then reordered with a **reranking** step so that the language model receives the most relevant context possible.

# Table of Contents
- [ 1 - Introduction](#1)
  - [ 1.1 RAG architecture overview](#1-1)
  - [ 1.2 Importing the necessary libraries](#1-2)
- [ 2 - Loading the dataset](#2)
- [ 3 - Chunking the BBC News Articles](#3)
- [ 4 - Creating the BBC News Vector Collection in Weaviate](#4)
- [ 5 - Loading the Collection](#5)
- [ 6 - Retrieval Methods](#6)
  - [ 6.1 Metadata Filtering](#6-1)
  - [ 6.2 Keyword Search](#6-2)
  - [ 6.3 Semantic Search](#6-3)
  - [ 6.4 Hybrid Search](#6-4)
  - [ 6.5 Semantic Search with Reranking](#6-5)
- [ 7 - Generating the final prompt](#7)
- [ 8 - LLM call](#8)
- [ 7 - Comparison between different approaches](#9)

<a id='1'></a>
## 1 - Introduction

---

<a id='1-1'></a>
### 1.1 RAG Architecture Overview

Below is a simplified representation of a Retrieval-Augmented Generation (RAG) pipeline:

<div align="center">
  <img src="../src/assets/rag_overview.png" alt="RAG Overview" width="60%">
</div>

The system follows a structured workflow. A retriever first identifies the most relevant documents from the dataset based on a user query. The retrieved content is then formatted and injected into an augmented prompt. This enriched prompt is finally passed to the language model to generate a grounded response.

To evaluate the impact of retrieval, responses generated with the RAG pipeline are compared against responses produced without additional retrieved context. This comparison highlights how external knowledge influences factual accuracy and relevance in the model’s output.

<a id='1-2'></a>
### 1.2 Importing the necessary libraries

In [1]:
import os
import sys
from pathlib import Path
sys.path.extend([
    str(Path.cwd().parent),
    str(Path.cwd().parent / "src"),
])
import data
import joblib
from utils.rag_weaviate_core import (
    print_object_properties,
    prepare_weaviate_objects,
    filter_by_metadata,
    semantic_search_retrieve,
    bm25_retrieve,
    hybrid_retrieve,
    semantic_search_with_reranking,
    generate_final_prompt,
    llm_call,
    display_widget,
    )
from utils.rag_core import (
    generate_with_single_input,
)   
import weaviate
from weaviate.classes.config import Configure, Property, DataType 
from tqdm import tqdm
from weaviate.util import generate_uuid5

resource module not available on Windows


<a id='2'></a>
## 2 - Loading the dataset

In [2]:
bbc_data = joblib.load(os.path.join(os.path.dirname(data.__file__), "bbc_data.joblib"))

In [3]:
print_object_properties(bbc_data[0:2]) 

article_content: Justin Welby speaks on BBC Radio 4's Today programme as part of a special show guest edited by Dame ...(truncated)
description: The Archbishop of Canterbury urges politicians to "forswear wedge issues" and avoid divisive topics.
guid: https://www.bbc.co.uk/news/uk-67844356
link: https://www.bbc.co.uk/news/uk-67844356?at_medium=RSS&at_campaign=KARANGA
pubDate: 2024-01-01 00:00:04
title: Justin Welby: Political leaders should treat opponents as human beings

article_content: Rishi Sunak at a lung cancer screening programme in Nottingham in June 2023 Almost three million peo...(truncated)
description: Record numbers are being tested for cancer but treatment targets are still being missed in England.
guid: https://www.bbc.co.uk/news/health-67841348
link: https://www.bbc.co.uk/news/health-67841348?at_medium=RSS&at_campaign=KARANGA
pubDate: 2024-01-01 00:09:56
title: Almost three million tested for cancer in England



## 3 - Chunking the BBC News Articles

In this step, the BBC News articles are split into overlapping fixed-size chunks in order to prepare them for vectorization and insertion into Weaviate.

In [4]:
chunked_bbc_data = prepare_weaviate_objects(
    articles=bbc_data,
    chunk_size=120,
    overlap_fraction=0.2
)

print("Number of generated objects:", len(chunked_bbc_data))
print(chunked_bbc_data[0].keys())
print(chunked_bbc_data[0]["chunk_index"])
print(chunked_bbc_data[0]["chunk"][:300])

Number of generated objects: 80827
dict_keys(['title', 'description', 'article_content', 'chunk', 'chunk_index', 'guid', 'link', 'pubDate'])
0
Justin Welby speaks on BBC Radio 4's Today programme as part of a special show guest edited by Dame Emma Warmsley The Archbishop of Canterbury has urged politicians not to treat their opponents as enemies but fellow human beings. Speaking to the BBC, the Most Rev Justin Welby warned Britain's leader


## 4 - Creating the BBC News Vector Collection in Weaviate

In this section, we connect to **Weaviate** and create a **vector collection** for the **BBC News** data. Each object in the collection stores the news content together with its embedding vector, which will be used later for retrieval.

### 4.1 Connecting to a local Weaviate instance

In [2]:
# Connecting to a local Weaviate instance
client = weaviate.connect_to_local(
    host="localhost",
    port=8080,
    grpc_port=50051
)
# Checking that the instance is ready
print("Weaviate is ready:", client.is_ready())

Weaviate is ready: True


### 4.2 Creating the Weaviate collection

In [6]:
if client.collections.exists("bbc_collection"):
    client.collections.delete("bbc_collection")

In [7]:
# Define the vectorizer
vectorizer_config = [Configure.NamedVectors.text2vec_transformers(
                name="main_vector", # This is the name you will need to access the vectors of the objects in your collection
                source_properties=['title', 'chunk', 'description'], # which properties should be used to generate a vector, they will be appended to each other when vectorizing
                vectorize_collection_name = False, # This tells the client to not vectorize the collection name. 
                                                   # If True, it will be appended at the beginning of the text to be vectorized
                inference_url="http://host.docker.internal:5000/", # Since we are using an API based vectorizer, you need to pass the URL used to make the calls 
                                                       # This was setup in our Flask application
            )]

collection_name = "bbc_collection" 

# Create the collection
collection = client.collections.create(
    name=collection_name,
    vectorizer_config=vectorizer_config,
    properties=[
        Property(name="title", vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="description", vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="article_content", vectorize_property_name=False, data_type=DataType.TEXT),
        Property(name="chunk", vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="chunk_index", data_type=DataType.INT),
        Property(name="guid", data_type=DataType.TEXT),
        Property(name="link", data_type=DataType.TEXT),
        Property(name="pubDate", data_type=DataType.DATE),
    ]
)

print(f"Collection '{collection_name}' created successfully.") 

c:\Users\berka\OneDrive\Bureau\sysrag\.venv\Lib\site-packages\weaviate\warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(


Collection 'bbc_collection' created successfully.


In [8]:
print(collection)

<weaviate.Collection config={
  "name": "Bbc_collection",
  "description": null,
  "generative_config": null,
  "inverted_index_config": {
    "bm25": {
      "b": 0.75,
      "k1": 1.2
    },
    "cleanup_interval_seconds": 60,
    "index_null_state": false,
    "index_property_length": false,
    "index_timestamps": false,
    "stopwords": {
      "preset": "en",
      "additions": null,
      "removals": null
    }
  },
  "multi_tenancy_config": {
    "enabled": false,
    "auto_tenant_creation": false,
    "auto_tenant_activation": false
  },
  "object_ttl_config": null,
  "properties": [
    {
      "name": "title",
      "description": null,
      "data_type": "text",
      "index_filterable": true,
      "index_range_filters": false,
      "index_searchable": true,
      "nested_properties": null,
      "tokenization": "word",
      "vectorizer_config": null,
      "vectorizer": null,
      "vectorizer_configs": {
        "text2vec-transformers": {
          "skip": false,
     

### 4.3 Inserting the chunked data into Weaviate

In [9]:
# Set up a batch process with specified fixed size and concurrency
with collection.batch.fixed_size(batch_size=3, concurrent_requests=1) as batch:
    # Iterate over a subset of the dataset
    for document in tqdm(chunked_bbc_data[0:5]): # tqdm is a library to show progress bars
            # Generate a UUID based on the article_content text for unique identification
            uuid = generate_uuid5(document)
            print(document['title'])
            # print(uuid)
            # Add the object to the batch with properties and UUID. 
            # properties expects a dictionary with the keys being the properties.
            batch.add_object(
                properties=document,
                uuid=uuid,
            ) 

100%|██████████| 5/5 [00:00<00:00, 1251.13it/s]
c:\Users\berka\OneDrive\Bureau\sysrag\.venv\Lib\site-packages\weaviate\warnings.py:256: UserWarning: Con002: You are using the datetime object 2024-01-01 00:00:04 without a timezone. The timezone will be set to UTC.
            To use a different timezone, specify it in the datetime object. For example:
            datetime.datetime(2021, 1, 1, 0, 0, 0, tzinfo=datetime.timezone(-datetime.timedelta(hours=2))).isoformat() = 2021-01-01T00:00:00-02:00
            
  warnings.warn(


Justin Welby: Political leaders should treat opponents as human beings
Justin Welby: Political leaders should treat opponents as human beings
Justin Welby: Political leaders should treat opponents as human beings
Justin Welby: Political leaders should treat opponents as human beings
Justin Welby: Political leaders should treat opponents as human beings


## 5 - Loading the Collection

In [3]:
collection = client.collections.get("bbc_collection")

In [5]:
object = collection.query.fetch_objects(limit = 1, include_vector = True).objects[0]
print("Printing the properties (some will be truncated due to size)")
print_object_properties(object.properties)
print("Vector: (truncated)",object.vector['main_vector'][0:15])
print("Vector length: ", len(object.vector['main_vector']))

Printing the properties (some will be truncated due to size)
article_content: Justin Welby speaks on BBC Radio 4's Today programme as part of a special show guest edited by Dame ...(truncated)
chunk: war, and to seek to make peace. And we trust in God who promises peace, with justice." Mr Welby will...(truncated)
chunk_index: 4
description: The Archbishop of Canterbury urges politicians to "forswear wedge issues" and avoid divisive topics.
guid: https://www.bbc.co.uk/news/uk-67844356
link: https://www.bbc.co.uk/news/uk-67844356?at_medium=RSS&at_campaign=KARANGA
pubDate: 2024-01-01 00:00:04+00:00
title: Justin Welby: Political leaders should treat opponents as human beings

Vector: (truncated) [-0.05622948333621025, -0.04269353300333023, 0.06212451681494713, -0.051641955971717834, -0.002049860544502735, 0.012005348689854145, 0.058329857885837555, 0.04453718662261963, -0.009442934766411781, -0.034247949719429016, -0.04981986805796623, -0.03561501204967499, -0.023058203980326653, 0.027732

## 6 - Retrieval Methods

### 6.1 - Metadata Filtering

In [8]:
# Let's get an example
res = filter_by_metadata('title', ['Justin Welby'], collection, limit = 2)
for x in res:
    print_object_properties(x)

article_content: Justin Welby speaks on BBC Radio 4's Today programme as part of a special show guest edited by Dame ...(truncated)
chunk: Justin Welby speaks on BBC Radio 4's Today programme as part of a special show guest edited by Dame ...(truncated)
chunk_index: 0
description: The Archbishop of Canterbury urges politicians to "forswear wedge issues" and avoid divisive topics.
guid: https://www.bbc.co.uk/news/uk-67844356
link: https://www.bbc.co.uk/news/uk-67844356?at_medium=RSS&at_campaign=KARANGA
pubDate: 2024-01-01 00:00:04+00:00
title: Justin Welby: Political leaders should treat opponents as human beings

article_content: Justin Welby speaks on BBC Radio 4's Today programme as part of a special show guest edited by Dame ...(truncated)
chunk: about the process in which we can differ hugely, but not destructively." He noted that, up to the la...(truncated)
chunk_index: 2
description: The Archbishop of Canterbury urges politicians to "forswear wedge issues" and avoid divisive topi

### 6.2 - Semantic Search

In [9]:
print_object_properties(semantic_search_retrieve(query = 'Tell me about the last Taylor Swift show', collection = collection, top_k = 2))

article_content: Justin Welby speaks on BBC Radio 4's Today programme as part of a special show guest edited by Dame ...(truncated)
chunk: for BBC Radio 4's Today programme, which is being guest edited by Dame Emma Walmsley, chief executiv...(truncated)
chunk_index: 1
description: The Archbishop of Canterbury urges politicians to "forswear wedge issues" and avoid divisive topics.
guid: https://www.bbc.co.uk/news/uk-67844356
link: https://www.bbc.co.uk/news/uk-67844356?at_medium=RSS&at_campaign=KARANGA
pubDate: 2024-01-01 00:00:04+00:00
title: Justin Welby: Political leaders should treat opponents as human beings

article_content: Justin Welby speaks on BBC Radio 4's Today programme as part of a special show guest edited by Dame ...(truncated)
chunk: "forswear wedge issues" that render their opponents their enemies. "Actually, we have to say: 'My op...(truncated)
chunk_index: 3
description: The Archbishop of Canterbury urges politicians to "forswear wedge issues" and avoid divisive topi

### 6.3 - BM25 Search

In [10]:
print_object_properties(bm25_retrieve('Tell me about the last Taylor Swift show', collection, top_k = 2))

article_content: Justin Welby speaks on BBC Radio 4's Today programme as part of a special show guest edited by Dame ...(truncated)
chunk: about the process in which we can differ hugely, but not destructively." He noted that, up to the la...(truncated)
chunk_index: 2
description: The Archbishop of Canterbury urges politicians to "forswear wedge issues" and avoid divisive topics.
guid: https://www.bbc.co.uk/news/uk-67844356
link: https://www.bbc.co.uk/news/uk-67844356?at_medium=RSS&at_campaign=KARANGA
pubDate: 2024-01-01 00:00:04+00:00
title: Justin Welby: Political leaders should treat opponents as human beings

article_content: Justin Welby speaks on BBC Radio 4's Today programme as part of a special show guest edited by Dame ...(truncated)
chunk: for BBC Radio 4's Today programme, which is being guest edited by Dame Emma Walmsley, chief executiv...(truncated)
chunk_index: 1
description: The Archbishop of Canterbury urges politicians to "forswear wedge issues" and avoid divisive topi

### 6.4 - Hybrid Search

In [11]:
print_object_properties(hybrid_retrieve('Tell me about the last Taylor Swift show', collection, top_k = 2))

article_content: Justin Welby speaks on BBC Radio 4's Today programme as part of a special show guest edited by Dame ...(truncated)
chunk: for BBC Radio 4's Today programme, which is being guest edited by Dame Emma Walmsley, chief executiv...(truncated)
chunk_index: 1
description: The Archbishop of Canterbury urges politicians to "forswear wedge issues" and avoid divisive topics.
guid: https://www.bbc.co.uk/news/uk-67844356
link: https://www.bbc.co.uk/news/uk-67844356?at_medium=RSS&at_campaign=KARANGA
pubDate: 2024-01-01 00:00:04+00:00
title: Justin Welby: Political leaders should treat opponents as human beings

article_content: Justin Welby speaks on BBC Radio 4's Today programme as part of a special show guest edited by Dame ...(truncated)
chunk: about the process in which we can differ hugely, but not destructively." He noted that, up to the la...(truncated)
chunk_index: 2
description: The Archbishop of Canterbury urges politicians to "forswear wedge issues" and avoid divisive topi

### 6.5 - Semantic Search with Reranking

In [12]:
# Set a query
query = 'Tell me about the conflicts in Latin America'
# Get the results from a search (in this case the hybrid search)
results = semantic_search_with_reranking(query, collection = collection, top_k = 2, rerank_property = 'chunk')

In [13]:
results

[{'chunk': '"forswear wedge issues" that render their opponents their enemies. "Actually, we have to say: \'My opponent is never my enemy. My opponent is always my fellow human being. We disagree profoundly, we disagree on incredibly important things, but they\'re human\'." In his new year\'s message - which will be broadcast on BBC One at 12:55 GMT on Monday - Mr Welby will speak of hope for a "peaceful 2024" and reflect on the wars between Israel and Hamas, and Russia and Ukraine. He will say: "Jesus Christ tells us to stand with those suffering because of war, and to seek to make peace. And we trust in God who promises peace, with justice." Mr Welby will also speak of the',
  'description': 'The Archbishop of Canterbury urges politicians to "forswear wedge issues" and avoid divisive topics.',
  'pubDate': datetime.datetime(2024, 1, 1, 0, 0, 4, tzinfo=datetime.timezone.utc),
  'guid': 'https://www.bbc.co.uk/news/uk-67844356',
  'title': 'Justin Welby: Political leaders should treat o

## 7 - Generating the final prompt

In [6]:
prompt = generate_final_prompt("Tell me the economic situation of the US in 2024.", top_k = 5, retrieve_function = semantic_search_retrieve, use_rerank = False, rerank_property = 'title', collection = collection)

In [7]:
print(prompt) 

Answer the user query below. There will be provided additional information for you to compose your answer. The relevant information provided is from 2024 and it should be added as your overall knowledge to answer the query, you should not rely only on this information to answer the query, but add it to your overall knowledge.The news data is ordered by relevance.Query: Tell me the economic situation of the US in 2024.
2024 News: Title: Justin Welby: Political leaders should treat opponents as human beings, Chunk: for BBC Radio 4's Today programme, which is being guest edited by Dame Emma Walmsley, chief executive of pharmaceutical company GSK. Globally, 2024 will see more than half of the world's population heading to the polls, with votes scheduled in nations including in the US, Taiwan, India, Pakistan, South Africa and South Sudan. The next general election in the UK must be held before the end of January 2025, but it is up to Prime Minister Rishi Sunak to choose when to call it. Mr

## 8 - LLM call

In [5]:
query = "Tell me about United States and Brazil's relationship over the course of 2024. Provide links for the resources you use in the answer."

In [ ]:
# Result with reranked results
print(llm_call(query = query, 
               top_k = 5, 
               retrieve_function = hybrid_retrieve, 
               collection = collection
               ))

Okay, here's an overview of the United States and Brazil's relationship in 2024, incorporating information about global trends and perspectives as of early 2024, along with relevant resources.

**United States and Brazil: 2024 Overview**

In 2024, the relationship between the United States and Brazil is characterized by a complex mix of cooperation, competition, and evolving geopolitical dynamics. While historically strong economic and diplomatic ties exist, the relationship has seen shifts in recent years, influenced by changes in leadership and domestic priorities in both countries.

**Key Areas of Interaction in 2024:**

*   **Economic Ties:** Brazil remains a significant trading partner for the U.S., particularly in commodities like soybeans, iron ore, and petroleum. Trade figures show a continued, albeit fluctuating, volume of exchange. The U.S. is a major investor in Brazil, and Brazilian investment in the U.S. also remains substantial. However, trade disputes and differing econo

In [4]:
display_widget(
    llm_call_func=llm_call,
    collection=collection,
    semantic_search_retrieve=semantic_search_retrieve,
    bm25_retrieve=bm25_retrieve,
    hybrid_retrieve=hybrid_retrieve,
    semantic_search_with_reranking=semantic_search_with_reranking,
)

c:\Users\berka\OneDrive\Bureau\sysrag\.venv\Lib\site-packages\traitlets\traitlets.py:1385: DeprecationWarning: Passing unrecognized arguments to super(Layout).__init__(gap='20px').
object.__init__() takes exactly one argument (the instance to initialize)
This is deprecated in traitlets 4.2.This error will be raised in a future release of traitlets.
  warn(
c:\Users\berka\OneDrive\Bureau\sysrag\.venv\Lib\site-packages\traitlets\traitlets.py:1385: DeprecationWarning: Passing unrecognized arguments to super(Layout).__init__(overflow_y='auto').
object.__init__() takes exactly one argument (the instance to initialize)
This is deprecated in traitlets 4.2.This error will be raised in a future release of traitlets.
  warn(


c:\Users\berka\OneDrive\Bureau\sysrag\.venv\Lib\site-packages\traitlets\traitlets.py:1385: DeprecationWarning: Passing unrecognized arguments to super(Layout).__init__(border_radius='10px').
object.__init__() takes exactly one argument (the instance to initialize)
This is deprecated in traitlets 4.2.This error will be raised in a future release of traitlets.
  warn(


HTML(value='\n        <div style="\n            background: linear-gradient(90deg, #f8f9fa 0%, #eef3f8 100%);\…